In [7]:
import pandas as pd
import os
from pathlib import Path
import json
import glob
from rapidfuzz import process, fuzz
import pandas as pd

CHAPTER_NUM = 4

entities = pd.read_csv(f"../booknlp_out/pride_and_prejudice_{CHAPTER_NUM}.entities", sep="\t")

N = entities[(entities["cat"] == "PER")].groupby("COREF").size()
#sort by N descending
N = N.sort_values(ascending=False)

print(f"Character counts for chapter {CHAPTER_NUM}")
print("-" * 40)
print("char id\t entity count\t instances")
for coref, count in N.items():
    if count > 1:
        if not entities[(entities["COREF"] == coref) & (entities["prop"]=="PROP")]["text"].empty:
            print(f"{coref}\t {count}\t\t", entities[(entities["COREF"] == coref) & (entities["prop"]=="PROP")]["text"].unique())

Character counts for chapter 4
----------------------------------------
char id	 entity count	 instances
5	 49		 ['Mr. Bingley' 'Bingley']
3	 31		 ['Jane']
4	 13		 ['Elizabeth' 'Dear Lizzy']
9	 9		 ['Miss Bennet']
8	 7		 ['Darcy']
6	 4		 ['Miss Bingley']
7	 4		 ['Mrs. Hurst']


In [8]:
results_dir = "../results"
os.makedirs(results_dir, exist_ok=True)

In [9]:
#store proper entity name counts per chapter in a csv file with instance column
for i in range(1, 62):
    entities = pd.read_csv(f"../booknlp_out/pride_and_prejudice_{i}.entities", sep="\t")
    N = entities[(entities["cat"] == "PER")].groupby("COREF").size()
    N = N.sort_values(ascending=False)
    rows = []
    for coref, count in N.items():
        if count > 1:
            if not entities[(entities["COREF"] == coref) & (entities["prop"]=="PROP")]["text"].empty:
                instances = ", ".join(entities[(entities["COREF"] == coref) & (entities["prop"]=="PROP")]["text"].unique())
                rows.append((coref, count, instances))
    df = pd.DataFrame(rows, columns=["char_id", "entity_count", "instances"])
    df.to_csv(f"../results/chapter_{i}_character_counts.csv", index=False)
    

In [10]:
# Attached is the spreadsheet for my manual / analog tagging *all* of PRIDE AND PREJUDICE. 
# Make sure to look at the data under the tab labeled “All instances” because it is much more complete and therefore much more accurate. 
# It’s broken down by  —character —chapter —component. 
# Components are abbreviated as: 
#     N for name 
#     A for action [a score of 1 for each verb of physical action — actions related to speech, thought, and feeling tagged separately] 
#     B for backstory, we agreed we won’t include this moving forward  
#     C for communication [directly quoted dialogue, paraphrased speech, and letters; a score of 1 for each reported block of speech] 
#     FID — free indirect discourse, we agreed we won’t include this moving forward 
#     I for interiority — each verb expressing thought, feeling, intention, interpretation 
#     DN — description by narrator [score of 1 for each sentence] 
#     DC — discussion of a character by other characters. [If Elizabeth talks about Darcy, that is a score of 1 for Darcy under DC — I scored each sentence] We’ll need to discuss moving forward if we want to maintain different measures (sometimes words, sometimes sentences, sometimes blocks) for each component.  
#     For context here is a brief explanation of why I used variable units of measure.  
#     A and I: Someone is more active if they are described as running, skipping, and jumping than just running, and feeling more if they hope and fear rather than just hope, so I counted each verb.  
#     For DN and DC: when the narrator or a character is describing someone the basic unit is the sentence, not the word. (Also it is difficult to count long strings of words manually)  
#     However I can see that counting the number of words would also capture the amount of description which contributes to the importance of the character described.  
#     For C, an entire letter and speech contained between quotation marks (or the equivalent in paraphrase) seems like a good basic unit to count and for manual tagging it was easier to count blocks than sentences or words. 
#     (Interestingly I got similar results using this method to the results Tara Menon got counting total number of words in the attached article; I believe Menon also counted number of utterances, which was similar to what I was counting.). 
#     Looking ahead to an article, I think it would be good to measure C slightly differently than Menon so that we are testing her results rather than simply replicating them. 
#     N is straightforward but some characters have several name variants.

booknlp_out = Path("../booknlp_out")

ACTION = ("verb.motion", "verb.contact", "verb.creation")
INTERIOR = ("verb.cognition", "verb.emotion", "verb.perception")

rows = []

for book_path in sorted(booknlp_out.glob("*.book")):
    stem = book_path.stem
    chap = stem.split("_")[-1]

    with open(book_path) as f:
        data = json.load(f)

        quotes   = pd.read_csv(booknlp_out / f"{stem}.quotes", sep="\t", low_memory=False)
        entities = pd.read_csv(booknlp_out / f"{stem}.entities", sep="\t", low_memory=False)
        supers   = pd.read_csv(booknlp_out / f"{stem}.supersense", sep="\t", low_memory=False)

    # Only keep real person proper names
    entities = entities[(entities["cat"] == "PER") & (entities["prop"] == "PROP")].copy()

    # Build supersense lookup
    ss = dict(zip(supers["start_token"].astype(int), supers["supersense_category"].astype(str)))

    # --- basic counts from .book JSON ---
    N, A, I = {}, {}, {}
    for ch in data["characters"]:
        cid = int(ch["id"])
        # count proper mentions only
        N[cid] = sum(m["c"] for m in ch.get("mentions", {}).get("proper", []))

        # classify verbs
        a = i = 0
        for v in ch.get("agent", []):
            t = int(v["i"])
            cat = ss.get(t, "")
            if any(cat.startswith(x) for x in ACTION): a += 1
            elif any(cat.startswith(x) for x in INTERIOR): i += 1
        A[cid], I[cid] = a, i

    # --- quote-based counts (C, DC) ---
    C = quotes["char_id"].value_counts().astype(int).to_dict()

    DC = {}
    if not quotes.empty:
        ents = entities.copy()
        ents["start_token"] = ents["start_token"].astype(int)
        ents["end_token"] = ents["end_token"].astype(int)
        for _, q in quotes.iterrows():
            s, e, spk = int(q["quote_start"]), int(q["quote_end"]), int(q["char_id"])
            inside = ents[(ents["start_token"] >= s) & (ents["end_token"] <= e)]
            for c in set(inside["COREF"].astype(int)) - {spk}:
                DC[c] = DC.get(c, 0) + 1

    # --- DN: narrator mentions (outside quotes) ---
    DN = {}
    qspans = quotes[["quote_start", "quote_end"]].astype(int).values.tolist()
    def in_quote(a, b):
        for s, e in qspans:
            if a >= s and b <= e:
                return True
        return False

    for _, r in entities.iterrows():
        cid, a, b = int(r["COREF"]), int(r["start_token"]), int(r["end_token"])
        if not in_quote(a, b):
            DN[cid] = DN.get(cid, 0) + 1

    # --- character name map ---
    names = {}
    for ch in data["characters"]:
        cid = int(ch["id"])
        props = ch.get("mentions", {}).get("proper", [])
        if props:
            names[cid] = props[0]["n"]

    # fallback from entity text if missing
    for _, r in entities.iterrows():
        c = int(r["COREF"])
        if c not in names:
            names[c] = r["text"]

    # --- collect all characters that appear as named persons ---
    all_ids = set(N) | set(A) | set(I) | set(C) | set(DC) | set(DN)
    for cid in sorted(all_ids):
        rows.append({
            "Chapter": chap,
            "Character": names.get(cid, f"char_{cid}"),
            "N": N.get(cid, 0),
            "A": A.get(cid, 0),
            "I": I.get(cid, 0),
            "C": C.get(cid, 0),
            "DC": DC.get(cid, 0),
            "DN": DN.get(cid, 0)
        })

# --- Final table ---
df = pd.DataFrame(rows)
df = df[df["Character"].notna()]  # remove nulls
df = df.sort_values(["Chapter", "Character"])
df["Score"] = df[["N", "A", "I", "C", "DC", "DN"]].sum(axis=1)
df.to_csv("booknlp_sharon_counts.csv", index=False)

print("wrote booknlp_sharon_counts.csv")
print(df.head(10).to_string(index=False))

wrote booknlp_sharon_counts.csv
Chapter   Character  N  A  I  C  DC  DN  Score
      1        Jane  0  0  0  0   1   0      1
      1  Lady Lucas  0  0  0  0   1   0      1
      1       Lizzy  3  0  0  0   3   0      6
      1       Lydia  0  0  0  0   1   0      1
      1  Michaelmas  0  0  0  0   1   0      1
      1  Mr. Bennet  6  3  7 12   3   3     34
      1 Mr. Bingley  4  1  2  0   4   0     11
      1  Mr. Morris  0  0  0  0   1   0      1
      1   Mrs. Long  2  0  0  0   2   0      4
      1 Sir William  0  0  0  0   1   0      1


In [11]:
# Drop nulls and noise (e.g. 'char_15')
df = df[~df["Character"].str.startswith("char_")]

df["Score"] = df[["N", "A", "I", "C", "DC", "DN"]].sum(axis=1)

# Convert chapter labels to integer for proper numeric sort
df["Chapter"] = df["Chapter"].astype(int)

# Sort by chapter (numeric) and by descending Score
df = df.sort_values(["Chapter", "Score"], ascending=[True, False])

# reset index
df = df.reset_index(drop=True)

# Save and preview
df.to_csv("booknlp_sharon_counts_clean.csv", index=False)
print("wrote booknlp_sharon_counts_clean.csv")
print(df.groupby("Chapter").head(10))

wrote booknlp_sharon_counts_clean.csv
     Chapter    Character  N  A  I   C  DC  DN  Score
0          1   Mr. Bennet  6  3  7  12   3   3     34
1          1  Mr. Bingley  4  1  2   0   4   0     11
2          1        Lizzy  3  0  0   0   3   0      6
3          1    Mrs. Long  2  0  0   0   2   0      4
4          1         Jane  0  0  0   0   1   0      1
..       ...          ... .. .. ..  ..  ..  ..    ...
919       61        Kitty  2  2  1   0   0   2      7
920       61         Jane  3  0  0   0   0   3      6
921       61         Mary  2  0  1   0   0   2      5
922       61   Mr. Bennet  1  0  2   0   0   1      4
923       61  Mr. Bingley  2  0  0   0   0   2      4

[596 rows x 9 columns]


In [12]:
sharon = pd.read_excel("../data/manual/pride_and_prejudice.xlsx", sheet_name="ALL INSTANCES")
char_set = set(sharon['Character'].value_counts().index)

In [13]:
df

,Chapter,Character,N,A,I,C,DC,DN,Score
0,1,Mr. Bennet,6,3,7,12,3,3,34
1,1,Mr. Bingley,4,1,2,0,4,0,11
2,1,Lizzy,3,0,0,0,3,0,6
3,1,Mrs. Long,2,0,0,0,2,0,4
4,1,Jane,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...
927,61,Meryton,0,0,0,0,0,1,1
928,61,Mrs. Bingley,0,0,0,0,0,1,1
929,61,Pemberley,0,0,0,0,0,1,1
930,61,the Bingleys,0,0,0,0,0,1,1


In [14]:
booknlp_names = sorted(set(df['Character']))
print(booknlp_names)

sharon_set = set(x.lower() for x in char_set)
mine_set   = set(x.lower() for x in booknlp_names)

['Absence', 'All Meryton', 'Anne', 'April', 'Ashworth', 'Bingley', 'Bromley', 'CAROLINE BINGLEY', 'Captain Carter', 'Caroline', 'Catherine', 'Chamberlayne', 'Charles', 'Charlotte', 'Charlotte Lucas', 'Clarke', 'Colonel', 'Colonel F.', 'Colonel Fitzwilliam', 'Colonel Forster', 'Colonel Miller', 'Commerce', 'Darcy', 'Darcy,--', 'Dawson', 'De Bourgh', 'Dear ma’am', 'Denny', 'Derbyshire', 'Detection', 'Dovedale', 'E. GARDINER', 'Elizabeth', 'Elizabeth,--', 'Elizabeth,----', 'Elizabeth:--', 'FITZWILLIAM DARCY', 'February', 'Fordyce', 'General----', 'George Wickham', 'Georgiana', 'Georgiana Darcy', 'God', 'Good God', 'Good Heaven', 'Good Lord', 'Gretna Green', 'Haggerston', 'Happy', 'Harriet', 'Hertfordshire', 'Hill', 'Hunsford', 'Jane', 'John', 'June', 'Kent', 'Kitty', 'Kitty,--', 'LYDIA BENNET', 'Lady Anne', 'Lady Anne Darcy', 'Lady Catherine', 'Lady Catherine de Bourgh', 'Lady Lucas', 'Lady Metcalfe', 'Ladyship', 'Lizzy', 'Longbourn', 'Lord', 'Lord ----', 'Louisa', 'Lucas Lodge', 'Lydia',

In [15]:
missing_in_mine = sorted(sharon_set - mine_set)
extra_in_mine   = sorted(mine_set - sharon_set)

print(f"Found {len(sharon_set - set(missing_in_mine))} / {len(char_set)} of Sharon’s characters")
print(f"Missing ({len(missing_in_mine)}):", missing_in_mine[:10])
print(f"Extra BookNLP-only names ({len(extra_in_mine)}):", extra_in_mine[:10])

Found 45 / 55 of Sharon’s characters
Missing (10): ["darcy's father", 'hill (housekeeper)', 'housekeeper (mrs. reynolds)', 'mr. long', "mrs. bennet's brother", 'mrs. nicholls', 'the footman', 'the waiter', 'wickham sr.', 'young lucas']
Extra BookNLP-only names (129): ['_', 'a mr. philips', 'a mrs. younge , who was some time ago governess to miss darcy , and was dismissed from her charge', 'absence', 'all meryton', 'anne', 'april', 'ashworth', 'bingley', 'bromley']


In [16]:
print(extra_in_mine)

['_', 'a mr. philips', 'a mrs. younge , who was some time ago governess to miss darcy , and was dismissed from her charge', 'absence', 'all meryton', 'anne', 'april', 'ashworth', 'bingley', 'bromley', 'caroline', 'caroline bingley', 'catherine', 'chamberlayne', 'charles', 'charlotte', 'clarke', 'colonel', 'colonel f.', 'colonel miller', 'commerce', 'darcy', 'darcy,--', 'de bourgh', 'dear ma’am', 'denny', 'derbyshire', 'detection', 'dovedale', 'e. gardiner', 'elder miss bennets', 'elizabeth,--', 'elizabeth,----', 'elizabeth:--', 'february', 'fitzwilliam darcy', 'fordyce', 'general----', 'george wickham', 'georgiana', 'georgiana darcy', 'god', 'good god', 'good heaven', 'good lord', 'gretna green', 'haggerston', 'happy', 'harriet', 'hertfordshire', 'hill', 'hunsford', 'june', 'kent', 'kitty,--', 'lady anne', 'lady catherine', 'lady metcalfe', 'ladyship', 'lizzy', 'longbourn', 'lord', 'lord ----', 'louisa', 'lucas lodge', 'lydia bennet', 'mamma', 'maria', 'matlock', 'meryton', 'michaelmas

In [17]:
df

,Chapter,Character,N,A,I,C,DC,DN,Score
0,1,Mr. Bennet,6,3,7,12,3,3,34
1,1,Mr. Bingley,4,1,2,0,4,0,11
2,1,Lizzy,3,0,0,0,3,0,6
3,1,Mrs. Long,2,0,0,0,2,0,4
4,1,Jane,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...
927,61,Meryton,0,0,0,0,0,1,1
928,61,Mrs. Bingley,0,0,0,0,0,1,1
929,61,Pemberley,0,0,0,0,0,1,1
930,61,the Bingleys,0,0,0,0,0,1,1
